# CS232 — Kiểm tra khả thi: fine-tune Video SR trên degradation H.265

## Notebook này trả lời gì

Một câu hỏi duy nhất: **có nên dành 3 tháng cho hướng này không?**

Chạy hết trong **một phiên Kaggle 12 giờ**, và cho câu trả lời ở ba mốc:

| Mốc | Thời điểm | Nếu hỏng thì sao |
|---|---|---|
| **A** — model nạp được weights | ~30 phút | Dừng, báo lại để sửa kiến trúc |
| **B** — chạy được trên T4, biết giới hạn bộ nhớ | ~1 giờ | Nếu tràn ngay cả ở chuỗi ngắn → hướng này không khả thi trên Kaggle |
| **C** — fine-tune có cải thiện không | ~10 giờ | Nếu không cải thiện → 3 tháng nhiều khả năng cũng vậy |

## Vì sao chuyển sang ×4

Model video SR có sẵn (BasicVSR, RealBasicVSR) **chỉ hỗ trợ ×4**. Pipeline cũ của bạn là ×2 (540p→1080p), giờ thành **270p→1080p**.

Hệ quả: không so trực tiếp được với kết quả −67% ở notebook 2. Phải chạy lại baseline H.265 ở 270p trong chính notebook này.

Mặt được: ×4 là thiết lập chuẩn của mọi paper trong lĩnh vực, nên số của bạn so được với số công bố.

## Model dùng ở đây

**BasicVSR** — kiến trúc video SR cơ bản, gồm ba phần:

1. **SPyNet** — mạng ước lượng optical flow. Nhận hai frame liên tiếp, xuất ra cho mỗi pixel một cặp số (dx, dy) nói pixel đó dịch đi đâu. Đây là thứ thay thế motion vector từ codec mà bạn đã thử và thất bại.
2. **Truyền hai chiều** — chạy một lượt từ frame đầu đến cuối và một lượt ngược lại, mỗi lượt mang theo một tensor "trí nhớ" gọi là hidden state. Frame thứ 10 nhờ đó biết cả những gì xảy ra ở frame 50.
3. **Upsampling** — phóng to ×4 bằng pixel shuffle.

Khác model E của bạn ở chỗ: E xử lý từng ảnh độc lập, BasicVSR xử lý cả chuỗi và có cơ chế nhất quán thời gian **ngay trong kiến trúc**, không phải gắn thêm.

## Kế hoạch thời gian

| Phần | Việc | Ước tính |
|---|---|---|
| 1 | Cài đặt, dọn đĩa | 10 phút |
| 2 | Tải weights, **in cấu trúc key** | 10 phút |
| 3 | Định nghĩa kiến trúc, **nạp weights (MỐC A)** | 10 phút |
| 4 | **Đo bộ nhớ và tốc độ (MỐC B)** | 30 phút |
| 5 | Sinh dữ liệu H.265 ở ×4 | 60 phút |
| 6 | Baseline: H.265 thuần, BasicVSR chưa fine-tune | 40 phút |
| 7 | **Fine-tune (MỐC C)** | 6–7 giờ |
| 8 | Đánh giá cuối, kết luận | 40 phút |

Có đồng hồ chặn: nếu vượt ngân sách, phần huấn luyện tự dừng và chuyển sang đánh giá.

---
## Phần 1 — Kết nối Drive và cấu hình

### Cell dưới làm gì?
Mount Google Drive, kiểm tra GPU, tạo cấu trúc thư mục.

### Tại sao cần?
Colab xoá sạch mọi thứ khi phiên kết thúc. Drive là nơi duy nhất dữ liệu sống sót qua các phiên.

Phân chia: **Drive giữ thứ cần lâu dài** (video nguồn, checkpoint, kết quả), **ổ cục bộ `/content` giữ thứ tạm** (frame PNG, video trung gian). Lý do: đọc ghi hàng nghìn file PNG trên Drive rất chậm vì mỗi thao tác là một lệnh mạng.

### Chuẩn bị trên Drive trước khi chạy

```
MyDrive/CS232/
├── videos/     <- upload video 1080p vào đây
└── ckpt/       <- để trống, notebook tự ghi
```


In [ ]:
import os, shutil, subprocess, json, math, time, random, glob
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/CS232")
for sub in ("videos","ckpt","results"):
    (DRIVE/sub).mkdir(parents=True, exist_ok=True)

# thu muc lam viec cuc bo (nhanh) - mat khi het phien
WORK = Path("/content/work"); WORK.mkdir(exist_ok=True)
DATA=WORK/"vsr_data"; EV=WORK/"vsr_eval"; FG=WORK/"vsr_figs"
CK = DRIVE/"ckpt"                      # checkpoint ghi thang vao Drive
for d in (DATA,EV,FG): d.mkdir(parents=True, exist_ok=True)

def free_gb():
    st=os.statvfs("/content"); return st.f_bavail*st.f_frsize/1e9
print(f"Dia cuc bo con trong: {free_gb():.1f} GB")

import torch
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory/1e9
print(f"GPU: {gpu}, {vram:.1f} GB VRAM")
if "A100" in gpu:
    print("  Luu y: A100 tieu compute unit rat nhanh. T4 hoac L4 la du cho bai nay.")

T_START=time.time(); BUDGET_H=11.0
def elapsed_h(): return (time.time()-T_START)/3600
def check_budget(label=""):
    e=elapsed_h(); print(f"[{e:.2f}h / {BUDGET_H}h] {label}"); return e<BUDGET_H

EXTS = ("*.mp4","*.mkv","*.webm","*.mov","*.avi","*.MP4","*.MOV")
vids = sorted(sum([glob.glob(str(DRIVE/"videos"/e)) for e in EXTS], []))
print(f"\n{len(vids)} video trong Drive/CS232/videos/")
for v in vids[:10]: print(f"  {Path(v).name}  {os.path.getsize(v)/1e6:.0f} MB")
assert vids, "Chua co video. Upload file 1080p vao MyDrive/CS232/videos/"

In [ ]:
!pip install -q lpips 2>/dev/null
import numpy as np, cv2, torch.nn as nn, torch.nn.functional as F
import pandas as pd, matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE="cuda"
CFG = dict(
    scale=4, hr=(1920,1080), fps=24,
    n_clips=8, clip_len=5, val_clips=2,
    seq_train=7, patch_lr=64, batch=2,
    crf_list=[23,28,33,38],
    eval_bitrates=[200,400,800,1500],
    lr_gen=5e-5, train_hours=6.5, seed=42,
)
S=CFG["scale"]; HRW,HRH=CFG["hr"]; LRW,LRH=HRW//S, HRH//S
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
torch.backends.cudnn.benchmark=True

def run(cmd, t=1800):
    r=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)
    if r.returncode!=0: raise RuntimeError(f"{cmd}\n{r.stderr[-1200:]}")
    return r.stdout

print(run("ffmpeg -version | head -1").strip())
for lib in ["libx265","libx264"]:
    ok = subprocess.run(f"ffmpeg -hide_banner -encoders 2>/dev/null | grep -q {lib}",
                        shell=True).returncode==0
    print(f"{lib}: {'OK' if ok else 'THIEU'}")
print(f"\nLR {LRW}x{LRH} -> HR {HRW}x{HRH} (x{S})")
check_budget("setup xong")

---
## Phần 2 — Tải weights và in cấu trúc

### Cell dưới làm gì?
Tải checkpoint BasicVSR và SPyNet từ máy chủ OpenMMLab, rồi **in ra cấu trúc tên các tham số**.

### Tại sao cần?
Ở phần sau ta viết lại kiến trúc từ đầu thay vì cài thư viện `mmagic` — vì cài nó trên Kaggle hay vướng xung đột phiên bản `mmcv`, tốn hàng giờ và thường thất bại.

Viết lại thì phải đặt **đúng tên thuộc tính**, vì PyTorch sinh tên tham số từ tên biến. Đặt sai một chữ là không nạp được weights — lỗi này đã gặp ở notebook trước với `rdb1` và `b1`.

Cell in ra cây tên để đối chiếu. Nếu phần sau báo lệch key, bảng in ở đây cho biết chính xác phải sửa gì.

In [ ]:
URLS = {
 "basicvsr": "https://download.openmmlab.com/mmediting/restorers/basicvsr/"
             "basicvsr_reds4_20120409-0e599677.pth",
 "spynet":   "https://download.openmmlab.com/mmediting/restorers/basicvsr/"
             "spynet_20210409-c6c1bd09.pth",
}
paths={}
for name,url in URLS.items():
    p = CK/f"{name}.pth"
    if not p.exists():
        print(f"tai {name} ...", end=" ", flush=True)
        try:
            run(f'wget -q --timeout=180 -O {p} "{url}"', 600)
            print(f"{os.path.getsize(p)/1e6:.1f} MB")
        except Exception as e:
            print("LOI:", str(e)[:200])
    paths[name]=p

sd_raw = torch.load(paths["basicvsr"], map_location="cpu")
sd = sd_raw.get("state_dict", sd_raw)
sd = { (k[len("generator."):] if k.startswith("generator.") else k): v for k,v in sd.items() }
print(f"\n{len(sd)} tham so trong checkpoint BasicVSR\n")

# in cay ten theo nhom
from collections import OrderedDict
groups=OrderedDict()
for k in sd:
    top=k.split(".")[0]
    groups.setdefault(top, []).append(k)
for g,ks in groups.items():
    print(f"{g:24s} {len(ks):4d} tham so | vd: {ks[0]}")
print("\nVai shape quan trong:")
for k in list(sd)[:3] + [k for k in sd if "fusion" in k or "conv_last" in k][:4]:
    print(f"  {k:52s} {tuple(sd[k].shape)}")
check_budget("tai weights xong")

---
## Phần 3 — Kiến trúc BasicVSR (MỐC A)

### Cell dưới làm gì?
Viết lại toàn bộ kiến trúc BasicVSR và SPyNet bằng PyTorch thuần, rồi nạp weights.

### Các thành phần

**`SPyNet`** — mạng ước lượng optical flow theo kiểu kim tự tháp. Nó xử lý ảnh ở nhiều mức độ phân giải từ thô đến mịn: ước lượng chuyển động lớn ở mức thô trước, rồi tinh chỉnh dần ở các mức mịn hơn. Sáu mức, mỗi mức một mạng con 5 lớp.

**`ResidualBlockNoBN`** — khối residual không có batch normalization: `out = x + conv2(relu(conv1(x)))`. Bỏ batch norm vì với bài toán khôi phục ảnh, nó làm mất thông tin về độ sáng tuyệt đối.

**`ResidualBlocksWithInputConv`** — một lớp conv đầu vào rồi xếp chồng nhiều khối residual.

**`PixelShufflePack`** — phóng to ảnh: conv tạo ra `C × r²` kênh rồi sắp xếp lại thành ảnh lớn gấp `r` lần. Nhanh hơn và ít artifact hơn so với nội suy thông thường.

**`BasicVSRNet`** — ghép lại: tính flow giữa các frame liên tiếp, chạy lượt ngược rồi lượt xuôi, mỗi lượt warp hidden state theo flow rồi đưa qua khối residual, cuối cùng gộp hai lượt và phóng to.

### Tại sao đây là MỐC A
Nếu `load_state_dict` báo lệch key thì mình đặt tên sai ở đâu đó. Cell in ra chính xác key nào thiếu, key nào thừa — gửi lại cho mình là sửa được trong một lượt.

In [ ]:
def flow_warp(x, flow, interp="bilinear", pad="border", align=True):
    # x: (n,c,h,w), flow: (n,h,w,2) don vi pixel
    n,c,h,w = x.size()
    gy,gx = torch.meshgrid(torch.arange(h, device=x.device, dtype=x.dtype),
                           torch.arange(w, device=x.device, dtype=x.dtype), indexing="ij")
    grid = torch.stack((gx,gy), 2).float()                    # (h,w,2)
    g = grid.unsqueeze(0) + flow
    gx_ = 2.0*g[...,0]/max(w-1,1) - 1.0
    gy_ = 2.0*g[...,1]/max(h-1,1) - 1.0
    return F.grid_sample(x, torch.stack((gx_,gy_), dim=3), mode=interp,
                         padding_mode=pad, align_corners=align)

class ConvModule(nn.Module):
    # tuong duong mmcv ConvModule: sinh key dang '<ten>.conv.weight'
    def __init__(self, i, o, act=True):
        super().__init__()
        self.conv = nn.Conv2d(i,o,7,1,3)
        self.activate = nn.ReLU(inplace=True) if act else None
    def forward(self,x):
        x=self.conv(x)
        return self.activate(x) if self.activate is not None else x

class SPyNetBasicModule(nn.Module):
    # ten 'basic_module' va thu tu 0..4 phai khop checkpoint
    def __init__(self):
        super().__init__()
        self.basic_module = nn.Sequential(
            ConvModule(8,32), ConvModule(32,64), ConvModule(64,32),
            ConvModule(32,16), ConvModule(16,2, act=False))
    def forward(self, x): return self.basic_module(x)

class SPyNet(nn.Module):
    def __init__(self, n_level=6):
        super().__init__()
        self.basic_module = nn.ModuleList([SPyNetBasicModule() for _ in range(n_level)])
        self.register_buffer("mean", torch.tensor([.485,.456,.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([.229,.224,.225]).view(1,3,1,1))
    def compute_flow(self, ref, supp):
        n,_,h,w = ref.size()
        ref=[(ref-self.mean)/self.std]; supp=[(supp-self.mean)/self.std]
        for _ in range(5):
            ref.append(F.avg_pool2d(ref[-1],2,2,count_include_pad=False))
            supp.append(F.avg_pool2d(supp[-1],2,2,count_include_pad=False))
        ref=ref[::-1]; supp=supp[::-1]
        flow = ref[0].new_zeros(n,2,h//32,w//32)
        for lv in range(len(ref)):
            if lv==0: up=flow
            else:
                up = F.interpolate(flow, scale_factor=2, mode="bilinear", align_corners=True)*2.0
            flow = up + self.basic_module[lv](torch.cat([
                ref[lv], flow_warp(supp[lv], up.permute(0,2,3,1), pad="border"), up], 1))
        return flow
    def forward(self, ref, supp):
        n,_,h,w = ref.size()
        hu = int(math.ceil(h/32))*32; wu = int(math.ceil(w/32))*32
        ref = F.interpolate(ref, size=(hu,wu), mode="bilinear", align_corners=False)
        supp= F.interpolate(supp,size=(hu,wu), mode="bilinear", align_corners=False)
        flow = F.interpolate(self.compute_flow(ref,supp), size=(h,w),
                             mode="bilinear", align_corners=False)
        flow[:,0,:,:] *= float(w)/float(wu); flow[:,1,:,:] *= float(h)/float(hu)
        return flow

class ResidualBlockNoBN(nn.Module):
    def __init__(self, mid=64, res_scale=1.0):
        super().__init__()
        self.conv1=nn.Conv2d(mid,mid,3,1,1); self.conv2=nn.Conv2d(mid,mid,3,1,1)
        self.relu=nn.ReLU(inplace=True); self.res_scale=res_scale
    def forward(self,x): return x + self.conv2(self.relu(self.conv1(x)))*self.res_scale

class ResidualBlocksWithInputConv(nn.Module):
    def __init__(self, in_ch, out_ch=64, num_blocks=30):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,1,1), nn.LeakyReLU(0.1, inplace=True),
            nn.Sequential(*[ResidualBlockNoBN(out_ch) for _ in range(num_blocks)]))
    def forward(self,x): return self.main(x)

class PixelShufflePack(nn.Module):
    def __init__(self, in_ch, out_ch, scale, kernel=3):
        super().__init__()
        self.scale=scale
        self.upsample_conv = nn.Conv2d(in_ch, out_ch*scale*scale, kernel, padding=(kernel-1)//2)
    def forward(self,x): return F.pixel_shuffle(self.upsample_conv(x), self.scale)

class BasicVSRNet(nn.Module):
    def __init__(self, mid=64, num_blocks=30):
        super().__init__()
        self.spynet = SPyNet()
        self.backward_resblocks = ResidualBlocksWithInputConv(mid+3, mid, num_blocks)
        self.forward_resblocks  = ResidualBlocksWithInputConv(mid+3, mid, num_blocks)
        self.fusion   = nn.Conv2d(mid*2, mid, 1, 1, 0)
        self.upsample1= PixelShufflePack(mid, mid, 2, 3)
        self.upsample2= PixelShufflePack(mid, 64, 2, 3)
        self.conv_hr  = nn.Conv2d(64,64,3,1,1)
        self.conv_last= nn.Conv2d(64,3,3,1,1)
        self.img_upsample = nn.Upsample(scale_factor=4, mode="bilinear", align_corners=False)
        self.lrelu = nn.LeakyReLU(0.1, inplace=True)

    def compute_flow(self, lrs):
        n,t,c,h,w = lrs.size()
        a = lrs[:,:-1].reshape(-1,c,h,w)
        b = lrs[:,1:].reshape(-1,c,h,w)
        flows_backward = self.spynet(a,b).view(n,t-1,2,h,w)
        flows_forward  = self.spynet(b,a).view(n,t-1,2,h,w)
        return flows_forward, flows_backward

    def forward(self, lrs):
        n,t,c,h,w = lrs.size()
        flows_forward, flows_backward = self.compute_flow(lrs)
        outputs=[]
        feat = lrs.new_zeros(n,64,h,w)
        for i in range(t-1, -1, -1):                    # luot NGUOC
            if i < t-1:
                feat = flow_warp(feat, flows_backward[:,i].permute(0,2,3,1))
            feat = self.backward_resblocks(torch.cat([lrs[:,i], feat],1))
            outputs.append(feat)
        outputs = outputs[::-1]
        feat = torch.zeros_like(feat)
        for i in range(0, t):                           # luot XUOI
            if i > 0:
                feat = flow_warp(feat, flows_forward[:,i-1].permute(0,2,3,1))
            feat = self.forward_resblocks(torch.cat([lrs[:,i], feat],1))
            out = self.lrelu(self.fusion(torch.cat([outputs[i], feat],1)))
            out = self.lrelu(self.upsample1(out))
            out = self.lrelu(self.upsample2(out))
            out = self.lrelu(self.conv_hr(out))
            out = self.conv_last(out)
            outputs[i] = out + self.img_upsample(lrs[:,i])
        return torch.stack(outputs, dim=1)

# ---------- MOC A: nap weights ----------
net = BasicVSRNet()
own = net.state_dict()
missing   = [k for k in own if k not in sd]
unexpected= [k for k in sd  if k not in own]
shape_bad = [k for k in own if k in sd and tuple(own[k].shape)!=tuple(sd[k].shape)]
print(f"khop  : {len(own)-len(missing)}/{len(own)}")
print(f"thieu : {len(missing)}   thua: {len(unexpected)}   lech shape: {len(shape_bad)}")
if missing:    print("  vd thieu :", missing[:4])
if unexpected: print("  vd thua  :", unexpected[:4])
if shape_bad:  print("  vd lech  :", [(k, tuple(own[k].shape), tuple(sd[k].shape)) for k in shape_bad[:3]])

OK_A = (len(missing)==0 and len(shape_bad)==0)
if OK_A:
    net.load_state_dict(sd, strict=False)
    print("\n>>> MOC A DAT: nap weights thanh cong")
else:
    print("\n>>> MOC A HONG: gui ket qua cell nay de sua kien truc")
net = net.to(DEVICE).eval()
print(f"Tham so: {sum(p.numel() for p in net.parameters())/1e6:.2f}M")
check_budget("moc A")

---
## Phần 4 — Đo bộ nhớ và tốc độ (MỐC B)

### Cell dưới làm gì?
Chạy suy luận với chuỗi dài dần cho tới khi hết bộ nhớ, ghi lại ngưỡng.

### Tại sao cần?
Đây là rủi ro lớn nhất của cả hướng. Model video giữ hidden state cho **mọi frame trong chuỗi** cùng lúc, nên bộ nhớ tăng tuyến tính theo độ dài chuỗi. Có người báo cáo model đòi cấp phát 32 GB khi chạy trên chuỗi dài — T4 chỉ có 16 GB.

Kết quả cell này quyết định hai điều:
- **Chuỗi tối đa khi suy luận** — video dài hơn thì phải chia đoạn
- **Chuỗi tối đa khi huấn luyện** — huấn luyện cần thêm bộ nhớ cho gradient, thường chỉ chịu được khoảng một nửa so với suy luận

Nếu ngay cả chuỗi 3 frame ở 270p cũng tràn thì hướng này không chạy được trên Kaggle, và bạn biết điều đó sau một giờ thay vì sau ba tháng.

In [ ]:
def gpu_mem_gb(): return torch.cuda.max_memory_allocated()/1e9

rows=[]
print(f"Do o {LRW}x{LRH} (LR) -> {HRW}x{HRH} (HR)\n")
for T in [2,3,5,7,10,15,20,30]:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    try:
        x = torch.rand(1,T,3,LRH,LRW, device=DEVICE)
        torch.cuda.synchronize(); t0=time.perf_counter()
        with torch.no_grad(), torch.autocast("cuda", torch.float16):
            y = net(x)
        torch.cuda.synchronize(); dt=time.perf_counter()-t0
        rows.append(dict(seq=T, mem_GB=round(gpu_mem_gb(),2), giay=round(dt,2),
                         fps=round(T/dt,2), trang_thai="OK"))
        del x,y
    except torch.cuda.OutOfMemoryError:
        rows.append(dict(seq=T, mem_GB=np.nan, giay=np.nan, fps=np.nan, trang_thai="OOM"))
        torch.cuda.empty_cache(); break
    except Exception as e:
        rows.append(dict(seq=T, mem_GB=np.nan, giay=np.nan, fps=np.nan,
                         trang_thai=str(e)[:40])); break
MEM = pd.DataFrame(rows); display(MEM)

ok = MEM[MEM.trang_thai=="OK"]
MAX_INFER = int(ok.seq.max()) if len(ok) else 0
print(f"\nChuoi toi da khi SUY LUAN: {MAX_INFER} frame")
if len(ok):
    print(f"Toc do: {ok.fps.iloc[-1]:.2f} fps (muc tieu real-time la 24)")

# uoc luong cho huan luyen: patch nho hon nhieu nen thu truc tiep
print(f"\nThu HUAN LUYEN voi patch {CFG['patch_lr']}x{CFG['patch_lr']}, chuoi {CFG['seq_train']}:")
try:
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    net.train()
    xb = torch.rand(CFG["batch"], CFG["seq_train"], 3, CFG["patch_lr"], CFG["patch_lr"], device=DEVICE)
    yb = torch.rand(CFG["batch"], CFG["seq_train"], 3, CFG["patch_lr"]*S, CFG["patch_lr"]*S, device=DEVICE)
    with torch.autocast("cuda", torch.float16):
        loss = F.l1_loss(net(xb), yb)
    loss.backward()
    print(f"  OK, bo nho dinh: {gpu_mem_gb():.2f} GB")
    OK_B = True
except torch.cuda.OutOfMemoryError:
    print("  OOM -> giam patch_lr hoac seq_train")
    OK_B = False
net.zero_grad(set_to_none=True); net.eval(); torch.cuda.empty_cache()

print(f"\n>>> MOC B {'DAT' if OK_B and MAX_INFER>=3 else 'HONG'}")
check_budget("moc B")

---
## Phần 5 — Sinh dữ liệu H.265 ở tỉ lệ ×4

### Cell dưới làm gì?
Cắt clip, tạo ground truth 1080p, rồi sinh bản LR 270p nén H.265 ở nhiều mức CRF.

### Tại sao cần?
Đây là phần **giữ nguyên từ notebook 2** — phát hiện degradation mismatch vẫn đúng, chỉ đổi tỉ lệ từ ×2 sang ×4.

Điểm khác: LR giờ là 480×270 thay vì 960×540. Ở độ phân giải này artifact nén còn nặng hơn vì mỗi pixel phải gánh nhiều thông tin hơn.

CRF cũng dịch lên (23–38 thay vì 28–40) vì ảnh nhỏ hơn thì cùng một CRF cho bitrate thấp hơn.

In [ ]:

HRD=DATA/"hr"; HRD.mkdir(exist_ok=True)
clips=[]; cid=0
nper = max(1, math.ceil(CFG["n_clips"]/len(vids)))
for v in vids:
    if cid>=CFG["n_clips"]: break
    try: dur=float(run(f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{v}"').strip())
    except Exception: continue
    usable=dur-CFG["clip_len"]-2
    if usable<=2: continue
    for ss in np.linspace(1, usable, nper):
        if cid>=CFG["n_clips"]: break
        out=HRD/f"c{cid:02d}.mp4"
        if not out.exists():
            run(f'ffmpeg -y -loglevel error -ss {ss:.2f} -i "{v}" -t {CFG["clip_len"]} '
                f'-vf "scale={HRW}:{HRH}:flags=lanczos,fps={CFG["fps"]}" '
                f'-c:v libx264 -preset slow -crf 8 -pix_fmt yuv420p -an {out}', 600)
        clips.append(dict(id=cid, path=out)); cid+=1

random.Random(CFG["seed"]).shuffle(clips)
VAL=clips[:CFG["val_clips"]]; TRAIN=clips[CFG["val_clips"]:]
print(f"{len(clips)} clip | train {len(TRAIN)} | val {len(VAL)}")

def extract(video, outdir, vf=None):
    outdir=Path(outdir)
    if outdir.exists() and len(list(outdir.glob("*.png")))>0: return outdir
    outdir.mkdir(parents=True, exist_ok=True)
    vfa=f'-vf "{vf}"' if vf else ""
    run(f'ffmpeg -y -loglevel error -i {video} {vfa} {outdir}/%04d.png', 900)
    return outdir

STORE={}
for c in tqdm(clips, desc="sinh du lieu"):
    gt=extract(c["path"], DATA/f"gt{c['id']:02d}")
    for crf in CFG["crf_list"]:
        lrv=DATA/f"lr_c{c['id']:02d}_crf{crf}.mp4"
        if not lrv.exists():
            run(f'ffmpeg -y -loglevel error -i {c["path"]} '
                f'-vf "scale={LRW}:{LRH}:flags=lanczos" -c:v libx265 -preset medium '
                f'-crf {crf} -x265-params log-level=error -pix_fmt yuv420p -an {lrv}', 900)
        lrd=extract(lrv, DATA/f"lr_c{c['id']:02d}_crf{crf}")
        n=min(len(list(lrd.glob('*.png'))), len(list(Path(gt).glob('*.png'))))
        STORE[(c["id"],crf)]=dict(lr=lrd, gt=gt, n=n)

print(f"\n{len(STORE)} to hop (clip, CRF) | dia con: {free_gb():.1f} GB")
check_budget("sinh du lieu xong")

---
## Phần 6 — Baseline trước khi fine-tune

### Cell dưới làm gì?
Đo chất lượng của hai mốc so sánh: H.265 1080p trực tiếp, và BasicVSR pretrained chưa fine-tune.

### Tại sao cần?
Không có hai con số này thì không biết fine-tune có giúp gì không.

Dự đoán dựa trên literature: BasicVSR pretrained sẽ **kém** trên video nén, vì nó được huấn luyện trên ảnh giảm mẫu bicubic sạch — đúng cùng loại degradation mismatch mà bạn đã phát hiện với Real-ESRGAN. Một paper đo BasicVSR chỉ đạt 25.93 dB ở CRF25 trong khi model chuyên cho video nén đạt 28.05 dB.

Nếu điều đó lặp lại ở đây thì fine-tune có nhiều dư địa để cải thiện.

In [ ]:
import lpips
lp = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
from skimage.metrics import structural_similarity

def to_y(b): return cv2.cvtColor(b, cv2.COLOR_BGR2YCrCb)[:,:,0].astype(np.float64)
def psnr_y(a,b):
    m=np.mean((to_y(a)-to_y(b))**2); return 99.0 if m==0 else 10*np.log10(255.0**2/m)
def ssim_y(a,b): return structural_similarity(to_y(a),to_y(b),data_range=255.0)
@torch.no_grad()
def lpd(a,b):
    t=lambda im: torch.from_numpy(np.ascontiguousarray(im[:,:,::-1]).astype(np.float32)/127.5-1
                                  ).permute(2,0,1)[None].to(DEVICE)
    return lp(t(a),t(b)).item()
def kbps_of(p):
    d=float(run(f'ffprobe -v error -show_entries format=duration -of csv=p=0 {p}').strip())
    return os.path.getsize(p)*8/d/1000

CHUNK = max(2, min(MAX_INFER, 10))
@torch.no_grad()
def vsr_infer(model, lr_dir, out_dir, chunk=CHUNK, overlap=1):
    out_dir=Path(out_dir); shutil.rmtree(out_dir, ignore_errors=True); out_dir.mkdir(parents=True)
    files=sorted(Path(lr_dir).glob("*.png")); N=len(files)
    model.eval(); t0=time.perf_counter(); i=0
    while i < N:
        j=min(i+chunk, N)
        s=max(0, i-overlap); e=min(N, j+overlap)          # dem de tranh dut doan
        imgs=[cv2.imread(str(f))[:,:,::-1].astype(np.float32)/255. for f in files[s:e]]
        x=torch.from_numpy(np.ascontiguousarray(np.stack(imgs))).permute(0,3,1,2)[None].to(DEVICE)
        with torch.autocast("cuda", torch.float16):
            y=model(x).float().clamp(0,1)[0]
        for k in range(i,j):
            a=y[k-s].permute(1,2,0).cpu().numpy()[:,:,::-1]
            cv2.imwrite(str(out_dir/files[k].name), (a*255).round().astype(np.uint8))
        del x,y; torch.cuda.empty_cache(); i=j
    return N/(time.perf_counter()-t0)

def eval_dir(fdir, gt_files, gt_temporal):
    ps,ss,lps,tl=[],[],[],[]; po=None
    for i,f in enumerate(sorted(Path(fdir).glob("*.png"))):
        if i>=len(gt_files): break
        o=cv2.imread(str(f)); g=cv2.imread(str(gt_files[i]))
        ps.append(psnr_y(o,g)); ss.append(ssim_y(o,g)); lps.append(lpd(o,g))
        if po is not None: tl.append(abs(lpd(o,po)-gt_temporal[i-1]))
        po=o
    return dict(psnr_y=np.mean(ps), ssim=np.mean(ss), lpips=np.mean(lps),
                tlp=np.mean(tl) if tl else np.nan)

def gt_temporal_of(gt_files):
    out=[]; pg=None
    for f in gt_files:
        g=cv2.imread(str(f))
        if pg is not None: out.append(lpd(g,pg))
        pg=g
    return out

def full_eval(model, tag, rows):
    for c in VAL:
        gt_files=sorted((DATA/f"gt{c['id']:02d}").glob("*.png"))
        gtt=gt_temporal_of(gt_files)
        for br in CFG["eval_bitrates"]:
            va=EV/f"A_c{c['id']}_{br}.mp4"
            if not va.exists():
                run(f'ffmpeg -y -loglevel error -i {c["path"]} -c:v libx265 -preset medium '
                    f'-b:v {br}k -maxrate {int(br*1.5)}k -bufsize {br*3}k '
                    f'-x265-params log-level=error -pix_fmt yuv420p -an {va}', 900)
            da=extract(va, EV/f"A_c{c['id']}_{br}")
            if not any(r["method"]=="A: H.265 1080p" and r["clip"]==c["id"] and r["target"]==br
                       for r in rows):
                rows.append(dict(clip=c["id"], method="A: H.265 1080p", target=br,
                                 kbps=kbps_of(va), fps=np.nan, **eval_dir(da, gt_files, gtt)))
            vl=EV/f"LR_c{c['id']}_{br}.mp4"
            if not vl.exists():
                run(f'ffmpeg -y -loglevel error -i {c["path"]} '
                    f'-vf "scale={LRW}:{LRH}:flags=lanczos" -c:v libx265 -preset medium '
                    f'-b:v {br}k -maxrate {int(br*1.5)}k -bufsize {br*3}k '
                    f'-x265-params log-level=error -pix_fmt yuv420p -an {vl}', 900)
            lrk=kbps_of(vl); dlr=extract(vl, EV/f"LR_c{c['id']}_{br}")
            od=EV/f"{tag}_c{c['id']}_{br}"
            f_=vsr_infer(model, dlr, od)
            rows.append(dict(clip=c["id"], method=tag, target=br, kbps=lrk, fps=f_,
                             **eval_dir(od, gt_files, gtt)))
    return rows

ROWS=[]
ROWS=full_eval(net, "V: BasicVSR pretrained", ROWS)
BASE=pd.DataFrame(ROWS); display(BASE.round(4))
BASE.to_csv(WORK/"vsr_baseline.csv", index=False)
check_budget("baseline xong")

---
## Phần 7 — Fine-tune trên degradation H.265 (MỐC C)

### Cell dưới làm gì?
Huấn luyện BasicVSR trên các cặp (LR nén H.265, HR sạch) đã sinh ở phần 5, có đồng hồ chặn theo ngân sách.

### Tại sao cần?
Đây là câu hỏi trung tâm: **áp dụng phát hiện degradation mismatch lên kiến trúc video có hiệu quả không?**

### Các lựa chọn kỹ thuật

**Đóng băng SPyNet trong 2000 vòng đầu.** Mạng flow đã được huấn luyện tốt; để nó tự do ngay từ đầu khi các phần khác còn chưa ổn định sẽ phá hỏng chất lượng ước lượng chuyển động. Đây cũng là cách cấu hình gốc của BasicVSR làm (`fix_iter=5000`).

**Charbonnier loss** thay vì L1 thuần: `sqrt((x-y)² + ε²)`. Nó gần giống L1 nhưng khả vi ở mọi điểm nên ổn định hơn khi huấn luyện.

**Đồng hồ chặn.** Vòng lặp kiểm tra thời gian đã trôi và tự dừng khi hết ngân sách, lưu checkpoint. Nhờ vậy notebook luôn chạy hết tới phần đánh giá thay vì bị Kaggle cắt giữa chừng.

**Không dùng GAN loss.** Ở bước kiểm tra khả thi này, ta chỉ cần biết fine-tune có cải thiện không. Thêm adversarial làm huấn luyện phức tạp và khó chẩn đoán.

In [ ]:
class SeqDS(Dataset):
    def __init__(self, store, ids, seq=7, patch=64):
        self.items=[]
        for (cid,crf),v in store.items():
            if cid not in ids: continue
            for t0 in range(0, v["n"]-seq):
                self.items.append((cid,crf,t0))
        self.store=store; self.seq=seq; self.p=patch
    def __len__(self): return len(self.items)
    def __getitem__(self,i):
        cid,crf,t0=self.items[i]; v=self.store[(cid,crf)]
        p=self.p
        probe=cv2.imread(str(v["lr"]/f"{t0+1:04d}.png"))
        if probe is None: return self[(i+1)%len(self)]
        h,w=probe.shape[:2]
        if h<p or w<p: return self[(i+1)%len(self)]
        y=random.randint(0,h-p); x=random.randint(0,w-p)
        fh = random.random()<0.5; fv = random.random()<0.5
        to_t=lambda a: torch.from_numpy(np.ascontiguousarray(a[:,:,::-1]).astype(np.float32)/255.).permute(2,0,1)
        lrs,gts=[],[]
        for k in range(self.seq):
            lr=cv2.imread(str(v["lr"]/f"{t0+k+1:04d}.png"))
            gt=cv2.imread(str(Path(v["gt"])/f"{t0+k+1:04d}.png"))
            if lr is None or gt is None: return self[(i+1)%len(self)]
            a=lr[y:y+p, x:x+p]; b=gt[y*S:(y+p)*S, x*S:(x+p)*S]
            if fh: a,b=a[:,::-1],b[:,::-1]
            if fv: a,b=a[::-1],b[::-1]
            lrs.append(to_t(a)); gts.append(to_t(b))
        return torch.stack(lrs), torch.stack(gts)

tr=SeqDS(STORE, {c["id"] for c in TRAIN}, CFG["seq_train"], CFG["patch_lr"])
dl=DataLoader(tr, batch_size=CFG["batch"], shuffle=True, num_workers=2,
              pin_memory=True, drop_last=True, persistent_workers=True)
print(f"{len(tr)} chuoi huan luyen")

def charbonnier(x,y,eps=1e-12): return torch.sqrt((x-y)**2+eps).mean()

net.train()
for p_ in net.spynet.parameters(): p_.requires_grad_(False)   # dong bang giai doan dau
opt=torch.optim.Adam([p_ for p_ in net.parameters() if p_.requires_grad], lr=CFG["lr_gen"])
scaler=torch.amp.GradScaler("cuda")
FIX_ITER=2000
CKF=CK/"basicvsr_h265.pt"; it0=0; hist=[]
if CKF.exists():
    s=torch.load(CKF, map_location=DEVICE)
    net.load_state_dict(s["net"]); opt.load_state_dict(s["opt"]); it0=s["it"]; hist=s["hist"]
    print(f"resume tu {it0}")

t_train0=time.time(); it=it0; di=iter(dl)
pbar=tqdm(desc="fine-tune")
while True:
    if (time.time()-t_train0)/3600 > CFG["train_hours"]:
        print(f"\nHet ngan sach huan luyen ({CFG['train_hours']}h), dung o iteration {it}"); break
    if elapsed_h() > BUDGET_H - 1.2:
        print(f"\nGan het ngan sach tong, dung o iteration {it}"); break
    if it == FIX_ITER:
        for p_ in net.spynet.parameters(): p_.requires_grad_(True)
        opt.add_param_group({"params": list(net.spynet.parameters()), "lr": CFG["lr_gen"]*0.25})
        print(f"\n[it {it}] mo dong bang SPyNet")
    try: lrs,gts=next(di)
    except StopIteration: di=iter(dl); lrs,gts=next(di)
    lrs,gts=lrs.to(DEVICE,non_blocking=True), gts.to(DEVICE,non_blocking=True)
    with torch.autocast("cuda", torch.float16):
        loss=charbonnier(net(lrs), gts)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    it+=1; pbar.update(1)
    if it%50==0: pbar.set_postfix(loss=f"{loss.item():.4f}", h=f"{elapsed_h():.1f}")
    if it%500==0:
        hist.append(dict(it=it, loss=loss.item(), h=elapsed_h()))
        torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), it=it, hist=hist), CKF)
pbar.close(); net.eval()
torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), it=it, hist=hist), CKF)
print(f"Da chay {it-it0} iteration")
if hist:
    h=pd.DataFrame(hist)
    plt.figure(figsize=(10,4)); plt.plot(h["it"], h["loss"], lw=3)
    plt.xlabel("iteration"); plt.ylabel("Charbonnier loss"); plt.grid(alpha=.3)
    plt.title("Fine-tune BasicVSR tren degradation H.265", fontsize=14)
    plt.tight_layout(); plt.savefig(FG/"finetune_loss.png", dpi=160); plt.show()
check_budget("moc C - huan luyen xong")

---
## Phần 8 — Đánh giá cuối và kết luận

### Cell dưới làm gì?
Chạy lại đánh giá với model đã fine-tune, so với hai baseline, rồi in kết luận.

### Cách đọc kết quả

| Quan sát | Ý nghĩa |
|---|---|
| Fine-tune thắng pretrained rõ trên LPIPS | Phát hiện degradation mismatch áp dụng được cho video SR — đáng đi tiếp |
| Fine-tune ngang hoặc kém hơn | Dữ liệu quá ít hoặc số vòng lặp quá ít; cần cân nhắc kỹ trước khi cam kết |
| tLP của cả hai thấp hơn model E cũ | Kiến trúc video giải quyết được nhấp nháy — đúng như kỳ vọng |
| Không model nào thắng H.265 ở bitrate thấp | Tỉ lệ ×4 có thể quá tham vọng ở mức bitrate này |

Con số quan trọng nhất là **LPIPS của bản fine-tune so với bản pretrained**. Đó là thứ nói lên phương pháp có chuyển được sang kiến trúc mới không.

In [ ]:
ROWS2=[r for r in ROWS]
ROWS2=full_eval(net, "V+FT: fine-tune H.265", ROWS2)
RES=pd.DataFrame(ROWS2); RES.to_csv(WORK/"vsr_results.csv", index=False)

M=RES.groupby(["method","target"])[["psnr_y","ssim","lpips","tlp","fps"]].mean().round(4)
print("=== Trung binh cac clip val ===")
display(M)

pre=RES[RES.method=="V: BasicVSR pretrained"]
ft =RES[RES.method=="V+FT: fine-tune H.265"]
a  =RES[RES.method=="A: H.265 1080p"]
d_lpips=100*(ft.lpips.mean()/pre.lpips.mean()-1)
d_psnr =ft.psnr_y.mean()-pre.psnr_y.mean()

fig,ax=plt.subplots(1,2,figsize=(14,5))
for m,g in RES.groupby("method"):
    g=g.groupby("target")[["lpips","tlp","kbps"]].mean().reset_index().sort_values("kbps")
    ax[0].plot(g.kbps,g.lpips,marker="o",lw=3,label=m)
    ax[1].plot(g.kbps,g.tlp,marker="o",lw=3,label=m)
for a_,t in zip(ax,["LPIPS (thap = tot)","tLP (thap = on dinh)"]):
    a_.set_xscale("log"); a_.set_xlabel("Bitrate (kbps)"); a_.set_title(t, fontsize=14)
    a_.grid(which="both",alpha=.3); a_.legend(fontsize=10)
plt.tight_layout(); plt.savefig(FG/"vsr_compare.png", dpi=160); plt.show()

print("\n"+"="*64)
print("KET LUAN KHA THI")
print("="*64)
print(f"MOC A - nap weights        : {'DAT' if OK_A else 'HONG'}")
print(f"MOC B - chay tren T4       : {'DAT' if OK_B else 'HONG'}  "
      f"(chuoi toi da {MAX_INFER} frame khi suy luan)")
print(f"MOC C - fine-tune cai thien: LPIPS {d_lpips:+.1f}%, PSNR {d_psnr:+.2f} dB")
print(f"\nToc do: {pre.fps.mean():.2f} fps (real-time can 24)")
print(f"So iteration da chay: {it}")
print(f"Tong thoi gian: {elapsed_h():.1f}h")
print()
if d_lpips < -5:
    print(">>> DANG DI TIEP. Fine-tune cai thien ro. Voi du lieu lon hon va")
    print("    nhieu iteration hon, cai thien nhieu kha nang con tang.")
elif d_lpips < 0:
    print(">>> CO TIN HIEU nhung yeu. Nut that nhieu kha nang la luong du lieu")
    print("    va so iteration. Can Vimeo-90K hoac REDS truoc khi ket luan.")
else:
    print(">>> CHUA CAI THIEN. Truoc khi cam ket 3 thang, can kiem tra:")
    print("    - So iteration da du chua (xem do thi loss con giam khong)")
    print("    - Du lieu co qua it khong (dang dung {} clip)".format(len(TRAIN)))

# gom file de tai ve
OUT=DRIVE/"results"; OUT.mkdir(exist_ok=True)
for f in WORK.glob("vsr_*.csv"): shutil.copy(f, OUT)
for f in FG.glob("*.png"): shutil.copy(f, OUT)
MEM.to_csv(OUT/"memory_profile.csv", index=False)
print("\nFile de tai ve:")
for f in sorted(OUT.iterdir()): print(f"  {f.name:36s} {os.path.getsize(f)/1e6:6.2f} MB")

---
## Nếu MỐC A hỏng

Nghĩa là tên thuộc tính trong kiến trúc viết lại không khớp checkpoint. Gửi lại phần in ra của cell phần 3 (danh sách key thiếu và thừa) — sửa được trong một lượt.

## Nếu MỐC B hỏng

Giảm `patch_lr` xuống 48, `seq_train` xuống 5, `batch` xuống 1 rồi chạy lại phần 4. Nếu vẫn tràn thì T4 không đủ cho kiến trúc này và cần cân nhắc lại hướng.

## Nếu MỐC C không cải thiện

Ba nguyên nhân có thể, theo thứ tự khả năng:

1. **Quá ít dữ liệu** — 6 clip so với 65 nghìn của Vimeo-90K
2. **Quá ít vòng lặp** — cấu hình gốc chạy 300 nghìn, ở đây được vài nghìn
3. **Learning rate** — 5e-5 lấy từ cấu hình gốc, nhưng batch nhỏ hơn nhiều nên có thể cần điều chỉnh

Đồ thị loss ở phần 7 cho biết nguyên nhân nào: nếu loss vẫn đang giảm đều khi hết giờ thì là (2); nếu nó phẳng sớm thì là (1) hoặc (3).